# Import libraries

In [1]:
import pyarrow as pa
import pandas as pd
import numpy as np
import os
import glob
import tqdm

# import data

## branches

In [2]:
branch_PE = pd.read_parquet("branch_PE.parquet")
branch_PAC = pd.read_parquet("branch_PAC.parquet")

In [3]:
tables = {
    "branch": ("branch_PE", "branch_PAC"),
    "customer": ("customer_PE", "customer_PAC"),
    "vehicle": ("vehicle_PE", "vehicle_PAC"),
}

In [4]:
for key, value in tables.items():
    df_PE = pd.read_parquet(f"{value[0]}.parquet")
    df_PAC = pd.read_parquet(f"{value[1]}.parquet")

    cols = df_PE.columns

    df_PE["StationBrand"] = "PE"
    df_PAC["StationBrand"] = "PAC"

    pd.concat([df_PE, df_PAC], axis=0).drop_duplicates(subset=cols).to_parquet(f"{key}.parquet")

## create invoice dfs

In [5]:
# 42m
df_list = []
for file in tqdm.tqdm(glob.glob(f'v_Invoice/*', recursive=True)):
    df = pd.read_parquet(file, engine="fastparquet",)
    df_list.append(df)

df_PE = pd.concat(df_list)
cols = df_PE.columns
df_PE["StationBrand"] = "PE"

df_list = []
for file in tqdm.tqdm(glob.glob(f'v_PAC_Invoice/*', recursive=True)):
    df = pd.read_parquet(file, engine="fastparquet",)
    df_list.append(df)

df_PAC = pd.concat(df_list)
df_PAC["StationBrand"] = "PAC"

100%|██████████| 59/59 [00:00<00:00, 72.78it/s]


In [6]:
# 42m
pd.concat([df_PE, df_PAC], axis=0).drop_duplicates(subset=cols).to_parquet("invoice.parquet", engine="fastparquet", index=False)

In [7]:
del df_PE
del df_PAC

In [8]:
df = pd.read_parquet("invoice.parquet")

In [6]:
df.groupby([df["InvoiceDate"].dt.year, df["InvoiceDate"].dt.month],).agg({"InvoiceID": "nunique"})

InvoiceID
InvoiceDate InvoiceDate           
2020        7                 3640
            8                22152
            9               154621
            10              253775
            11              248996
...                            ...
2025        3               261192
            4               269472
            5               310184
            6               179201
            12                   1

[61 rows x 1 columns]

In [9]:
del df

In [10]:
df_list = []
for file in tqdm.tqdm(glob.glob(f'v_InvoiceItems/*', recursive=True)):
    df = pd.read_parquet(file, engine="fastparquet")
    df_list.append(df)

df_PE = pd.concat(df_list)

del df_list

100%|██████████| 61/61 [00:26<00:00,  2.30it/s]


In [11]:
cols = df_PE.columns
df_PE["StationBrand"] = "PE"
# df_PE.to_parquet("invoiceitems_PE.parquet", index=False)

In [12]:
# 40m
df_PE.drop_duplicates(subset=cols).to_parquet("invoiceitems_PE.parquet", index=False, engine="fastparquet")

In [13]:
# 2m
df_list = []
for file in glob.glob(f'v_PAC_InvoiceItems/*', recursive=True):
    df = pd.read_parquet(file)
    df_list.append(df)

df_PAC = pd.concat(df_list)
df_PAC["StationBrand"] = "PAC"

In [14]:
df_PAC.drop_duplicates(subset=cols).to_parquet("invoiceitems_PAC.parquet", index=False, engine="fastparquet")

In [15]:
df_PAC = pd.read_parquet("invoiceitems_PAC.parquet", engine="fastparquet")

In [16]:
# 10m
pd.concat([df_PE, df_PAC], axis=0).drop_duplicates(subset=cols).to_parquet("invoiceitems.parquet", index=False, engine="fastparquet")

In [17]:
del df_PAC
del df_PE

In [18]:
df = pd.read_parquet("invoiceitems.parquet")

In [19]:
d

NameError: name 'd' is not defined

In [ ]:
df.columns

Index(['InvoiceID', 'InvoiceServiceID', 'ServiceBeforeDiscountAmount',
       'ServiceTotalDiscountAmount', 'ServiceBeforeTaxAmount',
       'ServiceTaxAmount', 'ServiceTotalAmount',
       'WithoutExtraServiceBeforeDiscountAmount',
       'WithoutExtraServiceTotalDiscountAmount',
       'WithoutExtraServiceBeforeTaxAmount', 'WithoutExtraServiceTaxAmount',
       'WithoutExtraServiceTotalAmount', 'ServiceItemGroupDefaultName',
       'ServiceItemDefaultName', 'ServiceItemCode',
       'ServiceManufacturerDefaultName', 'ItemBaseQuantity',
       'ServiceItemCostAmount', 'ItemBeforeDiscountAmount',
       'ItemTotalDiscountAmount', 'ItemBeforeTaxAmount', 'ItemTaxAmount',
       'ItemTotalAmount', 'WithoutExtraItemBeforeDiscountAmount',
       'WithoutExtraItemTotalDiscountAmount',
       'WithoutExtraItemBeforeTaxAmount', 'WithoutExtraItemTaxAmount',
       'WithoutExtraItemTotalAmount', 'ExtraItemBeforeDiscountAmount',
       'ExtraItemTotalDiscountAmount', 'ExtraItemBeforeTaxAmount',
 

In [17]:
df.groupby([df["InvoiceDate"].dt.year, df["InvoiceDate"].dt.month],).agg({"InvoiceID": "nunique"})

KeyError: 'InvoiceDate'